# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leiandrei/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd
import seaborn as sns

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

df = pd.read_csv("../data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*
- **Rule:** Flag any piece of content where the Predicted CTR (from the baseline model) is significantly higher than the Actual CTR (e.g., predicted is > 5% but actual is < 1%). Sort the queue by the largest deficit.

#### **Reason Codes:**
- `underperforming_serp_visibility`: Good rank, terrible clicks
- `high_vol_dilution`: Seach volume is too broad to capture clicks
- `intent_mismatch`: the content type does not satisfy its intent

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

,search_volume,avg_position,ctr,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,content_type_comparison article,content_type_keyword article,predicted_ctr,ctr_deficit
0,10.0,10.6,0.76,0,0,0,1,0,1,0.415940,-0.344060
1,90.0,20.3,0.05,0,1,0,0,0,1,0.066547,0.016547
2,0.0,36.5,0.09,0,1,0,0,0,1,0.140641,0.050641
3,10.0,6.2,0.49,1,0,0,0,0,1,0.355809,-0.134191
4,0.0,44.0,0.13,0,1,0,0,0,1,0.051905,-0.078095


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestRegressor

subset = df[['content_id', 'main_intent', 'content_type', 'search_volume', 'avg_position', 'ctr']]
subset = pd.get_dummies(subset, columns=['main_intent', 'content_type'], dtype='int')

X = subset.drop(['content_id', 'ctr'], axis=1)
y = subset['ctr']

rf = RandomForestRegressor(random_state=42).fit(X, y)
y_pred = rf.predict(X)

subset['predicted_ctr'] = y_pred
subset['ctr_deficit'] = subset['predicted_ctr'] - subset['ctr']

ranked_queue = subset.sort_values('ctr_deficit', ascending=False).reset_index(drop=True)
ranked_queue.head(20)

,content_id,search_volume,avg_position,ctr,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,content_type_comparison article,content_type_feedly article,content_type_keyword article,predicted_ctr,ctr_deficit
0,content_54f5b78e42f6,NaN,25.0,0.00,0,0,0,0,0,1,0,32.770817,32.770817
1,content_9952eb329b9c,NaN,25.0,0.00,0,0,0,0,0,1,0,32.770817,32.770817
2,content_d45265ca5145,NaN,0.7,0.00,0,0,0,0,0,1,0,27.387153,27.387153
3,content_8a7cfae5dfd8,0.0,98.0,0.00,0,1,0,0,0,0,1,23.885714,23.885714
4,content_e957954137ee,NaN,30.0,0.00,0,0,0,0,0,1,0,20.601587,20.601587
5,content_df644d79fad9,NaN,30.0,0.00,0,0,0,0,0,1,0,20.601587,20.601587
6,content_651845c6c973,NaN,30.0,0.00,0,0,0,0,0,1,0,20.601587,20.601587
7,content_48724397d104,NaN,2.5,5.26,0,0,0,0,0,0,1,25.650266,20.390266
8,content_8adc35ac7ead,NaN,1.0,0.00,0,0,0,0,0,1,0,19.123825,19.123825
9,content_f989e5fd85ed,NaN,1.0,0.00,0,0,0,0,0,1,0,19.123825,19.123825


In [17]:
ranked_queue.head(20).to_csv('../outputs/baseline_action_score.csv', index=False)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


#### **Content ID:** `content_48724397d104`
- **Action:** Audit SERP layout and rewrite Meta Title/Description to improve click appeal.
- **Reason Code:** `UNDERPERFORMING_TOP_RANK`
- **Confidence Note:** Moderate-High. The average position is stellar (2.5), but the actual CTR is only 5.26%. The baseline model expects a massive ~25.65% CTR for a keyword article ranking this high.
- **What would make it wrong:** If this specific query is heavily transactional and dominated by Sponsored Ads or a Local Pack that shoves our organic result below the fold, rendering a 25% CTR impossible.

#### **Content ID:** `content_ad5736686512`
- **Action:** Review competitor meta-data formatting; optimize title for higher relevance.
- **Reason Code:** `UNDERPERFORMING_TOP_RANK`
- **Confidence Note:** Moderate-High. Similar to the above, this keyword article holds a strong 2.5 average position, but is yielding only a 7.69% CTR against a 25.65% prediction.
- **What would make it wrong:** If the search intent shifted recently, meaning our content ranks well but no longer answers the user's primary question.

#### **Content ID:** `content_8a7cfae5dfd8`
- **Action:** Disregard/Deprioritize. Do not allocate content resources here.
- **Reason Code:** `INTENT_FATIGUE`
- **Confidence Note:** Very Low (Weak Pick). This page ranks at an abysmal Position 98.0 with 0 search volume, yet the model predicts a 23.88% CTR.
- **What would make it wrong:** Everything. A 23% CTR for a page on page 10 of Google is mathematically absurd. This is a model hallucination.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
ranked_queue.head(20)

,content_id,search_volume,avg_position,ctr,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,content_type_comparison article,content_type_feedly article,content_type_keyword article,predicted_ctr,ctr_deficit
0,content_54f5b78e42f6,NaN,25.0,0.00,0,0,0,0,0,1,0,32.770817,32.770817
1,content_9952eb329b9c,NaN,25.0,0.00,0,0,0,0,0,1,0,32.770817,32.770817
2,content_d45265ca5145,NaN,0.7,0.00,0,0,0,0,0,1,0,27.387153,27.387153
3,content_8a7cfae5dfd8,0.0,98.0,0.00,0,1,0,0,0,0,1,23.885714,23.885714
4,content_e957954137ee,NaN,30.0,0.00,0,0,0,0,0,1,0,20.601587,20.601587
5,content_df644d79fad9,NaN,30.0,0.00,0,0,0,0,0,1,0,20.601587,20.601587
6,content_651845c6c973,NaN,30.0,0.00,0,0,0,0,0,1,0,20.601587,20.601587
7,content_48724397d104,NaN,2.5,5.26,0,0,0,0,0,0,1,25.650266,20.390266
8,content_8adc35ac7ead,NaN,1.0,0.00,0,0,0,0,0,1,0,19.123825,19.123825
9,content_f989e5fd85ed,NaN,1.0,0.00,0,0,0,0,0,1,0,19.123825,19.123825


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
- **Weak Picks:** Several of the highest-ranked deficits in our queue (e.g., content_8a7cfae5dfd8 at avg_position 98.0, and content_54f5b78e42f6 at avg_position 25.0) are completely unrealistic. In real-world SEO, deep-ranked pages achieve virtually zero clicks. The Random Forest model is "hallucinating" 20% to 30% CTRs for these URLs because it likely memorized noisy, low-volume outliers in the training data and is incorrectly extrapolating that logic.

- **Missing Data:** 19 out of our top 20 rows are entirely missing their search_volume (NaN), and many completely lack a main_intent (all one-hot intent columns read 0). The model is being forced to predict on incomplete data branches, leading to severe inaccuracies at the very top of our action queue.

- **Leakage Check:** : Confirmed Clear. Looking at the dataset columns, there are absolutely no post-click behavioral metrics present. The model is genuinely trying to map discovery signals. The bad predictions are purely due to data quality and mathematical extrapolation, not target leakage.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.